<h1>1. Import library</h1>

In [1]:
############# Importing System Libraries #############
import sys
import os
import cv2

project_dir = "/data/atran16/ProteinClassification_3D"

############# Importing support Libraries #############
import json
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FormatStrFormatter
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

############# Importing datasets classes #############
sys.path.append(f"{project_dir}/utils/datasets")
from pdb_ds import test_tf, get_classes

############# Importing metrics #############
sys.path.append(f"{project_dir}/evaluations")
from SequentialSimilarity import similarity_score

############# Importing models #############
sys.path.append(f"{project_dir}")
from models import (
    load_Resnet,
    load_ConvNeXt,
    load_CoAtNet,
    load_EfficientNetV2,
    load_VIT_SizeT,
    load_RegNetY16GF,
    load_SwinV2B,
)

train_protein_path = f"{project_dir}/3D_PDB_5013/PNG126"
checkpoint_path    = f"{project_dir}/trained_results/24052026_train_126_30/Resnet152_smth_0/PDBRSTuan.pt"
test_root          = f"{project_dir}/3D_PDB_5013/testingDataFromProfessorSu"
image_size         = (224, 224)
topk               = (1, 3, 5, 10, 20, 50)

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = get_classes(train_protein_path)          # {idx: protein_id}
class_names_inv = {v: k for k, v in class_names.items()}  # {protein_id: idx}
configs = {"model": "Resnet152", "n_classes": len(class_names), "pretrained_path": ""}
print(f"Device: {device} | Classes: {len(class_names)}")


test_transform = test_tf(image_size)
MAX_K = max(topk)

Device: cuda | Classes: 5013


In [2]:
if "Resnet" in configs["model"]:
    model = load_Resnet(name=configs["model"], num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print(f"Loading {configs['model']} model successfully!\n")
elif configs["model"] == "ConvNeXt":
    model = load_ConvNeXt(num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading ConvNeXt model successfully!\n")
elif "CoAtNet" in configs["model"]:
    model = load_CoAtNet(name=configs["model"], num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading CoAtNet model successfully!\n")
elif "EfficientNetV2" in configs["model"]:
    model = load_EfficientNetV2(name=configs["model"], num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading EfficientNetV2 model successfully!\n")
elif configs["model"] == "MaxViT":
    model = load_VIT_SizeT(num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading MaxViT_SizeT model successfully!\n")
elif configs["model"] == "RegNetY16GF":
    model = load_RegNetY16GF(num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading RegNetY16GF model successfully!\n")
elif configs["model"] == "SwinV2B":
    model = load_SwinV2B(num_classes=configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading SwinV2B model successfully!\n")
else:
    raise ValueError(f"Unsupported model type: {configs['model']}")

state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state["net"])
model.to(device)
model.eval()

print(f"Model loaded from checkpoint: {checkpoint_path}")

Loading ResNet152 model successfully!

Loading Resnet152 model successfully!

Model loaded from checkpoint: /data/atran16/ProteinClassification_3D/trained_results/24052026_train_126_30/Resnet152_smth_0/PDBRSTuan.pt


In [3]:
########################################
# Required mappings (professor's pattern)
########################################
THRESHOLD    = 30
top_k        = 50
class_to_pdb = class_names  # {class_idx: pdb_id}

# Build test_items: (img_path, true_label_idx, true_pdb) sorted by filename number
test_root  = "/data/atran16/ProteinClassification_3D/3D_PDB_5013/testingDataFromProfessorSu_v2_229"
test_items = []

for protein in os.listdir(test_root):
    prot_path = os.path.join(test_root, protein)
    if not os.path.isdir(prot_path):
        continue
    true_pdb   = protein.upper()
    true_label = class_names_inv.get(true_pdb, -1)
    for fname in os.listdir(prot_path):
        if not fname.lower().endswith(".jpg") and not fname.lower().endswith(".png"):
            continue
        test_items.append((os.path.join(prot_path, fname), true_label, true_pdb))

test_items.sort(key=lambda x: (x[2], int(os.path.splitext(os.path.basename(x[0]))[0])))

# true_pdb_of: img_path -> ground-truth PDB ID
true_pdb_of = {img_path: true_pdb for img_path, _, true_pdb in test_items}

In [ ]:
########################################
# Evaluation loop (professor's pattern)
########################################
import csv

MAX_K      = max(topk)
y_true, y_pred = [], []
rows       = []

exact_hits  = {k: 0 for k in topk}
approx_hits = {k: 0 for k in topk}

log_csv_path = "/data/atran16/ProteinClassification_3D/visuallization/testVisuallizaion/SequentialSimilarityCheckResult.csv"
log_file     = open(log_csv_path, "w", newline="")
log_writer   = csv.writer(log_file)
log_writer.writerow(["image", "ground_truth", "best_sim", "match_via", "hit"])

for img_path, true_label, true_pdb in test_items:         # adapt to your loader
    img_num    = int(os.path.splitext(os.path.basename(img_path))[0])
    img_name   = f"{true_pdb}/{os.path.basename(img_path)}"

    img        = cv2.imread(img_path)[:, :, ::-1]  # BGR -> RGB
    img_tensor = test_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits       = model(img_tensor)
        probs        = torch.softmax(logits, dim=1).squeeze(0)
        topk_indices = torch.topk(probs, k=MAX_K).indices.tolist()

    # Professor's scan: break at first hit (>= THRESHOLD)
    best_sim, matched_pdb, first_hit_pos = 0.0, None, None

    for pos, cls in enumerate(topk_indices):
        pred_pdb = class_to_pdb[cls]

        try:
            sim = similarity_score(true_pdb, pred_pdb, verbose=0)
        except Exception as e:
            print(f"  [warn] similarity({true_pdb},{pred_pdb}) failed: {e}")
            sim = 0.0

        if sim > best_sim:
            best_sim, matched_pdb = sim, pred_pdb

        if sim >= THRESHOLD:
            first_hit_pos = pos
            break

    hit = first_hit_pos is not None

    # Write to CSV immediately
    log_writer.writerow([img_name, true_pdb, f"{best_sim:.2f}", matched_pdb, hit])
    log_file.flush()

    # Per-k metrics
    row = {"image": img_num, "protein": true_pdb}
    for k in topk:
        exact      = 1 if true_label in topk_indices[:k] else 0
        approx_hit = 1 if (first_hit_pos is not None and first_hit_pos < k) else 0

        exact_hits[k]  += exact
        approx_hits[k] += approx_hit

        row[f"exact_predictTop{k}"]   = exact
        row[f"countSimilarityTop{k}"] = approx_hit

    rows.append(row)

    pred_class = true_label if row[f"countSimilarityTop{top_k}"] else probs.argmax().item()
    y_true.append(true_label)
    y_pred.append(pred_class)

    print(f"[{img_num:>2}] {img_name:<12}  best_sim@top{MAX_K}={best_sim:5.1f}%  "
          f"match_via={matched_pdb}  hit={hit}")

log_file.close()
print(f"\nSaved per-image log -> {log_csv_path}")

# Professor's summary output
n = len(y_true)
print("\n" + "="*70)
for k in topk:
    print(f"Exact  Top-{k:<4}: {exact_hits[k]}/{n} ({exact_hits[k]/n:.2%})  |  "
          f"Approx Top-{k:<4} (>={THRESHOLD}% id): {approx_hits[k]}/{n} ({approx_hits[k]/n:.2%})")

[ 1] 7UZM/1.png    best_sim@top50=100.0%  match_via=7UZM  hit=True
[ 2] 7UZM/2.png    best_sim@top50=100.0%  match_via=7UZM  hit=True
[ 3] 7UZM/3.png    best_sim@top50=100.0%  match_via=7UZM  hit=True
[ 1] 8DNM/1.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 2] 8DNM/2.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 3] 8DNM/3.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 4] 8DNM/4.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 5] 8DNM/5.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 6] 8DNM/6.png    best_sim@top50= 22.5%  match_via=9OTM  hit=False
[ 7] 8DNM/7.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 8] 8DNM/8.png    best_sim@top50=100.0%  match_via=8DNM  hit=True
[ 9] 8DNM/9.png    best_sim@top50= 22.5%  match_via=9OTM  hit=False
[10] 8DNM/10.png   best_sim@top50=100.0%  match_via=8DNM  hit=True
[11] 8DNM/11.png   best_sim@top50=100.0%  match_via=8DNM  hit=True
[12] 8DNM/12.png   best_sim@top50=100.0%  match_via=8DNM  hi

In [ ]:
########################################
# Display table
########################################
cols_order = ["image", "protein"]
for k in topk:
    cols_order += [f"exact_predictTop{k}", f"countSimilarityTop{k}"]

df = pd.DataFrame(rows, columns=cols_order)

# Format image col as "<protein>/<N>.png"
df["image"] = df["protein"].astype(str) + "/" + df["image"].astype(str) + ".png"

# Sequential 1..N index (keep `image` as a regular column on the left)
df.index = range(1, len(df) + 1)
df.index.name = "order"

# Convert 0/1 columns to True/False
for k in topk:
    df[f"exact_predictTop{k}"] = df[f"exact_predictTop{k}"].astype(bool)
    df[f"countSimilarityTop{k}"] = df[f"countSimilarityTop{k}"].astype(bool)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

print(df)

             image protein  exact_predictTop1  countSimilarityTop1  exact_predictTop3  countSimilarityTop3  exact_predictTop5  countSimilarityTop5  exact_predictTop10  countSimilarityTop10  exact_predictTop20  countSimilarityTop20  exact_predictTop50  countSimilarityTop50
order                                                                                                                                                                                                                                                                           
1       7UZM/1.png    7UZM              False                False              False                False              False                False               False                 False               False                 False               False                 False
2       7UZM/2.png    7UZM              False                False              False                False              False                False               False               

In [ ]:
csv_path = "/data/atran16/ProteinClassification_3D/visuallization/testVisuallizaion/ExactNSimilarityCheckResult.csv"
df.to_csv(csv_path)
print(f"\nSaved to {csv_path}")


Saved to /data/atran16/ProteinClassification_3D/visuallization/testVisuallizaion/ExactNSimilarityCheckResult.csv
